In [1]:
!nvidia-smi

Sat Sep 14 12:17:24 2024       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.07              Driver Version: 550.90.07      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             26W /  250W |       0MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [25]:
import pandas as pd
df = pd.read_csv('/kaggle/input/ocr-data/data_100.csv')
df_test = pd.read_csv('/kaggle/input/ocr-data/data_10_test.csv')

In [26]:
import pandas as pd
import re

# Define unit mappings with support for different capitalizations
unit_mappings = {
    'g': 'gram',
    'G': 'gram',
    'gram': 'gram',
    'GRAM': 'gram',
    'cup': 'cup',
    'CUP': 'cup',
    'milligram': 'milligram',
    'MILLIGRAM': 'milligram',
    'mg': 'milligram',
    'MG': 'milligram',
    'kg': 'kilogram',
    'KG': 'kilogram',
    'kilogram': 'kilogram',
    'KILOGRAM': 'kilogram',
    'ounce': 'ounce',
    'OUNCE': 'ounce',
    'gallon': 'gallon',
    'GALLON': 'gallon',
    'volt': 'volt',
    'VOLT': 'volt',
    'watt': 'watt',
    'WATT': 'watt',
    'pound': 'pound',
    'POUND': 'pound',
    'millilitre': 'millilitre',
    'MILLILITRE': 'millilitre',
    'ml': 'millilitre',
    'ML': 'millilitre',
}

# Function to format text and fix missing spaces between units and numbers
def format_text(text):
    # Fix missing spaces between numbers and units
    pattern_units = re.compile(r'(\d+\.?\d*)(g|G|gram|GRAM|cup|CUP|milligram|MILLIGRAM|mg|MG|kg|KG|kilogram|KILOGRAM|ounce|OUNCE|gallon|GALLON|volt|VOLT|watt|WATT|pound|POUND|millilitre|MILLILITRE|ml|ML)', re.IGNORECASE)

    def replace_units(match):
        value = match.group(1)
        unit = match.group(2).lower()
        return f"{value} {unit_mappings.get(unit, unit)}"

    # Apply formatting for units
    formatted_text = pattern_units.sub(replace_units, text)

    # Fix missing spaces between words by inserting a space between lowercase-uppercase letter transitions or number-word transitions
    pattern_words = re.compile(r'([a-z])([A-Z])|(\d)([a-zA-Z])')

    def insert_space(match):
        return f"{match.group(1) or match.group(3)} {match.group(2) or match.group(4)}"

    # Apply formatting for missing spaces between words
    formatted_text = pattern_words.sub(insert_space, formatted_text)

    return formatted_text


In [55]:
# Apply the format_text function to the ocr_text column
df['ocr_text'] = df['ocr_text'].apply(format_text)
df_test['ocr_text'] = df_test['ocr_text'].apply(format_text)

In [56]:
df_test['entity_value'].loc[1]

'750.0 milligram'

In [57]:
df_test['ocr_text'].loc[1]

'Supplement Facts.  Serving Size 2 Capsules  Servings per Container 30  Milk  Thistle  Amount per Serving  % Daily Value  Bioactive Lipid Blend  750 mg  Beta sitosterol complex  (>95% sterols / 40% beta sitosterol)  Berberine HCI  Botanical Cholesterol Blend  750 mg  Artichoke  Milk thistle seed extract (80% silymarin).  Leaf  Globe artichoke leaf extract  Garlic bulb extract  *Daily value not established.  Other Ingredients: Vegetarian capsule (HPMC), vegetable  magnesium stearate,bamboo extract  Berberine  '

In [30]:
import pandas as pd
import re

# Function to find the starting position of entity_value in ocr_text
def find_entity_position(ocr_text, entity_value):
    # First, try to match the exact entity_value in the ocr_text
    entity_value_lower = entity_value.lower()
    ocr_text_lower = ocr_text.lower()
    
    # Try exact match
    exact_match = ocr_text_lower.find(entity_value_lower)
    if exact_match != -1:
        return exact_match

    # If exact match not found, extract numeric portion
    number_match = re.search(r'\d+\.?\d*', entity_value)
    
    if number_match:
        number_str = number_match.group()
        # Remove trailing zeros and decimal if it is an integer like '30.0' -> '30'
        normalized_number_str = str(float(number_str)).rstrip('0').rstrip('.') if '.' in number_str else number_str
        
        # Try to find the number in the ocr_text, check both decimal and non-decimal versions
        number_position = ocr_text_lower.find(normalized_number_str)
        if number_position != -1:
            return number_position

    # If neither exact match nor number match is found, look for any number in the ocr_text
    any_number_match = re.search(r'\d+\.?\d*', ocr_text_lower)
    if any_number_match:
        return any_number_match.start()

    # If no number is found in the ocr_text, return 0
    return 0


In [31]:
def df_to_dict(df):
    result = []
    for index, row in df.iterrows():
        # Creating a dictionary for each row
        entry = {
             "context": row['ocr_text'],
              "qas": [
            {
                "id" : index,
"is_impossible": False,
                "question": f"What is the {row['entity_name']}?",
                "answers": [
                    {"text":row['entity_value'], "answer_start":find_entity_position(row['ocr_text'], row['entity_value'])}
                ]
            }
        ]
        }
        result.append(entry)
    return result

# Convert dataframe to dictionary
train_data = df_to_dict(df)
test_data = df_to_dict(df_test)

In [32]:
import json

with open('amazon_data_train.json', 'w', encoding='utf-8') as f:
    json.dump(train_data, f, ensure_ascii=False, indent=4)

In [33]:
with open(r"amazon_data_train.json", "r") as read_file:
    train = json.load(read_file)

In [34]:
with open('amazon_data_test.json', 'w', encoding='utf-8') as f:
    json.dump(test_data, f, ensure_ascii=False, indent=4)

In [35]:
with open(r"amazon_data_test.json", "r") as read_file:
    test = json.load(read_file)

In [36]:
train[0]

{'context': "PROPOS  NATURE  INGREDIENT MENAGER  MULTI-USAGE  TERREDE  SOMMIERES  100%NATUREL  Argile 100% pure et naturelle, la terre de  Sommieres presente des proprietesabsorbantes  qui permettent le nettoyage a sec des taches  recalcitrantes sur toutes les surfaces (moquette,  tapis, parquet...). Elle est aussi efficace pour  desodoriser le linge.  Ingredient Bentonite  Dosage conseill Selon usage  ge fermobrde la chorte hm  100%  500 gram  LABORATOIRE PROPOS'NATUREE  ",
 'qas': [{'id': 0,
   'is_impossible': False,
   'question': 'What is the item_weight?',
   'answers': [{'text': '500.0 gram', 'answer_start': 426}]}]}

In [37]:
test[0]

{'context': 'Sonder  Angebot!  KIM JOHANSON  10%  Kaufe 3 Produkte von  uns und spare 10%  -20%  Kaufe 6 Produktevon  uns und spare 20%  Du kannst unsere Artikel ansehen, wenn du  auf Kim Johanson direkt beim Produkttitel klickst  Lege einfach die Artikel in den Warenkorb.Der Rabatt wird automatisch  an der Kasse abgezogen.Klicke dazu auf ,Zur Kasse gehen  Das Angebot gilt fur unser gesamtes Sortiment.  ',
 'qas': [{'id': 0,
   'is_impossible': False,
   'question': 'What is the item_weight?',
   'answers': [{'text': '3.0 pound', 'answer_start': 43}]}]}

In [38]:
!pip install simpletransformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.3/316.3 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 96.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 91.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.9/82.9 kB 5.9 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16161 sha256=b2aa3c5bb1422175b5614228fef2537a3a5baf0fcac869561db4a96b0e635ed0
  Stored in directory: /root/.cache/pip/wheels/1a/67/4a/ad4082dd7dfc30f2abfe4d80a2ed5926a506eb8a972b4767fa
Successfully built seqeval


In [39]:
import logging
from simpletransformers.question_answering import QuestionAnsweringModel, QuestionAnsweringArgs

In [40]:
#train_args are the parameters the QuestionAnswerringModel will use
train_args = {
    'overwrite_output_dir': True,
    "evaluate_during_training": True,
    "max_seq_length": 128,
    "num_train_epochs": 25, #25, after experimentations
    "evaluate_during_training_steps": 500,
    "save_model_every_epoch": False,
    "save_eval_checkpoints": False,
    "n_best_size":16, #batch_size is another important argument
    "train_batch_size": 16,
    "eval_batch_size": 16
}

In [41]:
model = QuestionAnsweringModel("bert",
                               "bert-large-cased",
                               args = train_args,
                               use_cuda=True)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Pl

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

/opt/conda/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [42]:
model.train_model(train, eval_data=test)

/opt/conda/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
convert squad examples to features:   0%|          | 0/100 [00:00<?, ?it/s]/opt/conda/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
Could not find answer: '500 gram' vs. '500.0 gram'
Could not find answer: '4 celd' vs. '1.0 cup'
Could not find answer: '0.709 g)Each' vs. '0.709 gram'
Could not find answer: '(0.709 g Each' vs. '0.709 gram'
Could not find answer: '1400 PLANTAGO' vs. '1400 milligram'
Could not find answer: '30 kilogram' vs. '30.0 kilogram'
Could not find answer: '<10 kg 10-15 kg 15-25 kg' vs. '10 kilogram to 15 kilogram'
Could not find answer: '0 Fe WEIGHT' vs. 

Epoch:   0%|          | 0/25 [00:00<?, ?it/s]

/opt/conda/lib/python3.10/site-packages/simpletransformers/question_answering/question_answering_model.py:697: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler()


Running Epoch 1 of 25:   0%|          | 0/2 [00:00<?, ?it/s]

/opt/conda/lib/python3.10/site-packages/simpletransformers/question_answering/question_answering_model.py:720: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():

convert squad examples to features: 100%|██████████| 10/10 [00:00<00:00, 277.54it/s]

add example index and unique id: 100%|██████████| 10/10 [00:00<00:00, 107546.26it/s]


...

Running Evaluation:   0%|          | 0/1 [00:00<?, ?it/s]

Running Epoch 25 of 25:   0%|          | 0/2 [00:00<?, ?it/s]


convert squad examples to features: 100%|██████████| 10/10 [00:00<00:00, 262.05it/s]

add example index and unique id: 100%|██████████| 10/10 [00:00<00:00, 95760.37it/s]


Running Evaluation:   0%|          | 0/1 [00:00<?, ?it/s]

(50,
 {'global_step': [2,
   4,
   6,
   8,
   10,
   12,
   14,
   16,
   18,
   20,
   22,
   24,
   26,
   28,
   30,
   32,
   34,
   36,
   38,
   40,
   42,
   44,
   46,
   48,
   50],
  'correct': [0,
   1,
   1,
   2,
   2,
   2,
   2,
   2,
   2,
   2,
   2,
   2,
   2,
   2,
   2,
   2,
   2,
   2,
   2,
   2,
   2,
   2,
   2,
   2,
   2],
  'similar': [1,
   1,
   1,
   0,
   0,
   0,
   1,
   2,
   2,
   2,
   3,
   4,
   4,
   2,
   2,
   1,
   1,
   1,
   1,
   1,
   1,
   1,
   2,
   2,
   2],
  'incorrect': [9,
   8,
   8,
   8,
   8,
   8,
   7,
   6,
   6,
   6,
   5,
   4,
   4,
   6,
   6,
   7,
   7,
   7,
   7,
   7,
   7,
   7,
   6,
   6,
   6],
  'train_loss': [4.899658203125,
   3.7615966796875,
   1.6181640625,
   0.6804046630859375,
   0.18989944458007812,
   0.17258930206298828,
   0.05676150321960449,
   0.016105711460113525,
   0.004370331764221191,
   0.0014912039041519165,
   0.0006176978349685669,
   0.0005909428000450134,
   0.00045783817768096924,


In [43]:
# Evaluate the model
result, texts = model.eval_model(test)

add example index and unique id: 100%|██████████| 10/10 [00:00<00:00, 77101.18it/s]


Running Evaluation:   0%|          | 0/1 [00:00<?, ?it/s]

In [44]:
print(result)

{'correct': 2, 'similar': 2, 'incorrect': 6, 'eval_loss': -6.13671875}


In [70]:
# Load model from training checkpoint
from simpletransformers.question_answering import QuestionAnsweringModel, QuestionAnsweringArgs
 
model = QuestionAnsweringModel("bert", "/kaggle/working/outputs/best_model")
 
 
#Make predictions with the model
to_predict = [
    {'context': '1.81 inch  1.02 inch  MATERIAL: Stainless steel  COLOR:  Steel color/Black/Golden  WEIGHT23 gram  WARM PROMPT  Manual measuring not rule out the possibility of  error.the above parameters are for reference  Chain length:20+2 inch  only.specific in kind prevail  ',
 'qas': [{'id': 3,
   'is_impossible': False,
   'question': 'What is the item_weight?',
}]}
]
 
answers, probabilities = model.predict(to_predict, n_best_size=2)
print(answers)

add example index and unique id: 100%|██████████| 1/1 [00:00<00:00, 10810.06it/s]


Running Prediction:   0%|          | 0/1 [00:00<?, ?it/s]

[{'id': 3, 'answer': ['23 gram', '1.81 inch 1.02 inch MATERIAL: Stainless steel COLOR: Steel color/Black/Golden WEIGHT23 gram']}]


In [69]:
test[3]

{'context': '1.81 inch  1.02 inch  MATERIAL: Stainless steel  COLOR:  Steel color/Black/Golden  WEIGHT23 gram  WARM PROMPT  Manual measuring not rule out the possibility of  error.the above parameters are for reference  Chain length:20+2 inch  only.specific in kind prevail  ',
 'qas': [{'id': 3,
   'is_impossible': False,
   'question': 'What is the item_weight?',
   'answers': [{'text': '23.0 gram', 'answer_start': 89}]}]}

In [71]:
import re

# Define unit mappings with support for different capitalizations
unit_mappings = {
    'g': 'gram',
    'G': 'gram',
    'gram': 'gram',
    'GRAM': 'gram',
    'cup': 'cup',
    'CUP': 'cup',
    'milligram': 'milligram',
    'MILLIGRAM': 'milligram',
    'mg': 'milligram',
    'MG': 'milligram',
    'kg': 'kilogram',
    'KG': 'kilogram',
    'kilogram': 'kilogram',
    'KILOGRAM': 'kilogram',
    'ounce': 'ounce',
    'OUNCE': 'ounce',
    'gallon': 'gallon',
    'GALLON': 'gallon',
    'volt': 'volt',
    'VOLT': 'volt',
    'watt': 'watt',
    'WATT': 'watt',
    'pound': 'pound',
    'POUND': 'pound',
    'millilitre': 'millilitre',
    'MILLILITRE': 'millilitre',
    'ml': 'millilitre',
    'ML': 'millilitre',
}

# Function to format only units in the text
def format_answer(text):
    # Pattern to find numbers followed by units
    pattern_units = re.compile(r'(\d+\.?\d*)(g|G|gram|GRAM|cup|CUP|milligram|MILLIGRAM|mg|MG|kg|KG|kilogram|KILOGRAM|ounce|OUNCE|gallon|GALLON|volt|VOLT|watt|WATT|pound|POUND|millilitre|MILLILITRE|ml|ML)', re.IGNORECASE)

    def replace_units(match):
        value = match.group(1)  # Extract the numeric value
        unit = match.group(2).lower()  # Extract and lowercase the unit
        return f"{value} {unit_mappings.get(unit, unit)}"  # Replace with standardized unit

    # Apply unit formatting
    formatted_text = pattern_units.sub(replace_units, text)
    
    return formatted_text



In [74]:
test = pd.read_csv('/kaggle/input/avengers-ml-dataset/test.csv')
test.head()

,index,image_link,group_id,entity_name
0,0,https://m.media-amazon.com/images/I/110EibNycl...,156839,height
1,1,https://m.media-amazon.com/images/I/11TU2clswz...,792578,width
2,2,https://m.media-amazon.com/images/I/11TU2clswz...,792578,height
3,3,https://m.media-amazon.com/images/I/11TU2clswz...,792578,depth
4,4,https://m.media-amazon.com/images/I/11gHj8dhhr...,792578,depth
